# Suite ERE — The POSIX ERE dialect

`Expressions.POSIX.ERE` models POSIX's extended regular expressions (IEEE Std 1003.1, XBD 9.4, its grammar 9.5.3), as
grep reads its patterns with `-E`, a newline-separated list of them, and awk its EREs, with C's escapes, and as C++'s
`std::regex` reads them under its `extended`, `awk` and `egrep` grammars. Its kinds are validated as its grammar has
them, bind nothing, cover POSIX.1's four editions, print as an ERE writes them, or awk, are refused by evaluation by
name, and are checked complete against XBD 9.5 kept as data.

In [ ]:
import { readFileSync as readGrammar } from "node:fs";
import { Framework } from "@mbse/expressions";
import * as BRE from "@mbse/expressions/POSIX/BRE";
import * as ERE from "../src/POSIX/ERE/index.js";
import { Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [B, R, RText] = [BRE.Expressions, ERE.Expressions, ERE.Text];

const store: any = new Builders.Store(); // the POSIX family's dialects, ERE's install first: the installs run in any order
for (const install of [Framework.install, ERE.install, BRE.install]) store.transaction(install);
store.transaction(ERE.define); // its editions

/** The term value `spec` writes, the root of `dialect`'s holder, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = R.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)))));
}

const validate = (spec: unknown): string[] => R.DIALECT.validate(term(spec));
const text = (spec: unknown, awk = false): string => RText.text(term(spec), awk);
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
/** A character, of its code point, and its spelling. */
const char = (c: string, spelling?: string) => (u: any) => u.character(
  (r: any) => spelling === undefined ? r.value(BigInt(c.codePointAt(0) as number)) : r.raw(spelling).value(BigInt(c.codePointAt(0) as number)));
const alt = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const capture = (...alternatives: unknown[]) => (u: any) => u.capturinggroup((r: any) => r.alternatives(alternatives));

/** A quantifier: `element{least,most}`, lazy where not `greedy`, as `spelling` writes it where given. */
function repeat(element: unknown, least: bigint, most?: bigint, greedy?: boolean, spelling?: string): unknown {
  return (u: any) => u.quantifier((r: any) => {
    r = r.min(least).element(element);
    r = most === undefined ? r : r.max(most);
    r = greedy === undefined ? r : r.greedy(greedy);
    return spelling === undefined ? r : r.raw(spelling);
  });
}

const cls = (elements: unknown[], negate = false) => (u: any) => u.characterclass((r: any) => r.negate(negate).elements(elements));
const span = (low: unknown, high: unknown) => (u: any) => u.characterclassrange((r: any) => r.min(low).max(high));
const chars = (kind: string) => (u: any) => u.characterset((r: any) => r.setKind(kind));
const dot = chars("any");
const anchor = (kind: string) => (u: any) => u.assertion((r: any) => r.assertionKind(kind));
const named = (kind: string, name: string) => (u: any) => u[kind]((r: any) => r.name(name)); // `[:alpha:]`, `[.a.]`, `[=a=]`
const patterns = (...specs: unknown[]) => (u: any) => u.patternlist((r: any) => r.patterns(specs));
const error = (source: string) => (u: any) => u.error((r: any) => r.text(source));
const missing = (kind: string) => (u: any) => u.missing((r: any) => r.kind(kind));
const abc = pattern(alt(char("a"), char("b")), alt(char("c"))); // `ab|c`

<a id="ERE-01"></a>
## ERE-01 · Its union and holder: its kinds and invalid syntax, no backreference, a branch of the framework's terms, reading its family's other dialect; a holder refining the framework's, round-tripped through JSON

In [ ]:
{
  assert(same(R.TAGS, ["patternlist", "pattern", "alternative", "capturinggroup", "quantifier", "characterclass",
    "characterclassrange", "characterset", "character", "assertion", "exclass", "collatingelement", "equivalence"])
    && R.HELPERS.length === 0 && R.SINCE.size === 0);
  assert(same([...R.DIALECT.kinds().keys()], union(R.DIALECT.union)) && same(union(R.DIALECT.union), [...R.TAGS, "error", "missing"]));
  assert(R.DIALECT.name() === "POSIX ERE" && R.DIALECT.tag === "posix_ere" && R.DIALECT.union === "Expressions.POSIX.ERE.Term");
  assert(union("Expressions.Framework.Term").includes("posix_ere") && union("Expressions.Framework.Dialect").includes("posix_ere"));
  assert(same(R.DIALECT.reads(), [B.DIALECT])); // its family's other dialect
  const Holder: any = store.registered("Expressions.POSIX.ERE.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(R.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const holder = store.transaction(() => store.Expressions.POSIX.ERE.Expression((e: any) => e.root(R.DIALECT.spec(abc))));
  assert(Validators.Validate(store)(Holder, holder).length === 0 && R.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === R.DIALECT.union && F.kindOf(F.term(holder))[0] === "pattern");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  raises(Error, () => store.transaction(() => // an ERE has no backreference (9.5.1)
    store.Expressions.POSIX.ERE.Expression((e: any) => e.root(R.DIALECT.spec((u: any) => u.backreference((r: any) => r.ref(1n)))))),
  "'backreference' is not a member of a union value");
  assert(R.DIALECT.validate(term((u: any) => u.backreference((r: any) => r.ref(1n)), B.DIALECT)).length === 0); // a BRE's, as it reads it
}

<a id="ERE-02"></a>
## ERE-02 · Its kinds: their properties in order, each term property its Term, a quantifier greedy or not; forms and make round-trip

In [ ]:
{
  const props = (name: string): string[] => [...store.registered(`Expressions.POSIX.ERE.${name}`).properties.keys()];
  assert(same(props("PatternList"), ["patterns"]) && same(props("Pattern"), ["alternatives"]) && same(props("CapturingGroup"), ["alternatives"]));
  assert(same(props("Alternative"), ["elements"]) && same(props("Quantifier"), ["raw", "min", "max", "greedy", "element"]));
  assert(same(props("CharacterClass"), ["negate", "elements"]) && same(props("CharacterClassRange"), ["min", "max"]));
  assert(same(props("CharacterSet"), ["setKind"]) && same(props("Character"), ["raw", "value"]) && same(props("Assertion"), ["assertionKind"]));
  assert(same(props("ExClass"), ["name"]) && same(props("CollatingElement"), ["name"]) && same(props("Equivalence"), ["name"]));
  const prop = (name: string, member: string): any => store.registered(`Expressions.POSIX.ERE.${name}`).properties.get(member).schema;
  const Term = store.registered(R.DIALECT.union);
  assert(prop("Quantifier", "element") === Term && prop("CharacterClassRange", "min") === Term && prop("CharacterClassRange", "max") === Term);
  assert(prop("PatternList", "patterns").element === Term && prop("Pattern", "alternatives").element === Term
    && prop("Alternative", "elements").element === Term);
  const quantified = F.form(term(repeat(cls([span(char("a"), char("z"))]), 1n, undefined, false)));
  assert(equal(quantified.attributes, new Map<string, unknown>([["min", 1n], ["greedy", false]]))
    && same(quantified.arguments.map((a) => F.kindOf(a)[0]), ["characterclass"]));
  for (const spec of [patterns(abc, pattern(alt(dot, anchor("end")))), repeat(repeat(anchor("start"), 1n), 0n, 1n),
    cls([named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), char("-")], true)]) {
    assert(F.same(term(R.DIALECT.make(F.form(term(spec)))), term(spec)));
  }
}

<a id="ERE-03"></a>
## ERE-03 · Its checks: what each kind's terms may be, as XBD 9.5.3 has them, an anchor anywhere, and its attributes' values

In [ ]:
{
  const cases: [unknown, string[]][] = [
    [patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]],
    [pattern(char("a")), ["a pattern's alternatives cannot hold a character"]],
    [alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]],
    [alt(named("exclass", "alpha")), ["an alternative's elements cannot hold an exclass"]],
    [capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]],
    [repeat(alt(), 0n), ["a quantifier's element cannot hold an alternative"]],
    [cls([dot]), ["a characterclass's elements cannot hold a characterset"]], // `.` within brackets is a character
    [span(dot, named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]],
    [repeat(char("a"), -1n), ["a quantifier's min cannot be negative, got -1"]],
    [repeat(char("a"), 3n, 2n), ["a quantifier's max cannot be less than its min, got 2 and 3"]],
    [(u: any) => u.quantifier((r: any) => r.min(1n)), ["a quantifier needs an element"]],
    [chars("word"), ["CharacterSet.setKind cannot be 'word'"]],
    [(u: any) => u.character((r: any) => r.value(-1n)), ["a character's value must be a code point, got -1"]],
    [anchor("lookahead"), ["Assertion.assertionKind cannot be 'lookahead'"]],
    [named("equivalence", "=e"), ["an equivalence's name cannot hold '='"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${validate(spec)} != ${problems}`);
  for (const spec of [abc, alt(char("a"), anchor("start"), anchor("end"), char("b")), // `a^$b`: an anchor anywhere (ERE_expression)
    capture(alt(anchor("start"), char("a"))), repeat(repeat(char("a"), 1n), 0n, 1n), // `ERE_expression ERE_dupl_symbol`
    repeat(anchor("start"), 0n), repeat(char("a"), 0n, undefined, false)]) { // POSIX.1-2024's repetition modifier
    assert(validate(spec).length === 0);
  }
}

<a id="ERE-04"></a>
## ERE-04 · grep -E's list of patterns, a line each; awk's EREs: their characters printed with C's escapes

In [ ]:
{
  const listed = patterns(pattern(alt(anchor("start"), char("a"))), abc);
  assert(validate(listed).length === 0 && text(listed) === "^a\nab|c"); // grep's pattern_list with -E (XCU grep)
  const awk = pattern(alt(char("\n"), char("\x07"), char("\x08"), char("\x01"), char("\x7f"), char("/"), char("+"),
    cls([char("\t"), char("+")])));
  assert(text(awk, true) === "\\n\\a\\b\\001\\177/\\+[\\t+]"); // XCU awk, Regular Expressions: C's escapes, in brackets too
  assert(text(awk) === "\n\x07\x08\x01\x7f/\\+[\t+]" && RText.ToText(term(awk)) === text(awk)); // an ERE writes a control character as itself
  assert(text(pattern(alt(char("\n", "\\012"))), true) === "\\012"); // a spelling as written
}

<a id="ERE-05"></a>
## ERE-05 · What it binds: nothing; a subexpression numbered, which no backreference resolves

In [ ]:
{
  assert(!store.registered(R.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
  const grouped = term(pattern(alt(capture(alt(char("a"))), repeat(capture(alt(char("b"))), 0n))));
  assert(SY.free(grouped, R.DIALECT).size === 0 && R.DIALECT.validate(grouped).length === 0);
  assert(same([...F.walk(grouped)].map((t) => F.kindOf(t)[0]), ["pattern", "alternative", "capturinggroup", "alternative",
    "character", "quantifier", "capturinggroup", "alternative", "character"]));
}

<a id="ERE-06"></a>
## ERE-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
{
  assert(same(validate(error("(")), ["invalid syntax '('"]) && text(error("(")) === "(");
  assert(same(validate(missing("atom")), ["missing atom"]) && text(alt(char("a"), missing("atom"))) === "a");
  assert(same(validate(pattern(alt(char("a"), error("{")))), ["argument 0: argument 1: invalid syntax '{'"]));
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "POSIX ERE error has no text to print");
}

<a id="ERE-07"></a>
## ERE-07 · Its editions: POSIX.1's four, as POSIX BRE's, each with every kind

In [ ]:
{
  const ere_ = store.Instances.Expressions.POSIX.ERE;
  assert(same([...FS.standards(ere_).values()].map((s) => FS.designationOf(s)), [
    "IEEE Std 1003.1-2001", "IEEE Std 1003.1-2008", "IEEE Std 1003.1-2017", "IEEE Std 1003.1-2024"]));
  assert(R.EDITIONS === B.EDITIONS && FS.find(ere_, "ISO/IEC 9945:2003") === FS.find(ere_, "POSIX.1-2001")); // C++17's and C++20's
  const names = ["PatternList", "Pattern", "Alternative", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "ExClass", "CollatingElement", "Equivalence"];
  assert([...FS.standards(ere_).values()].every((e) => same(FS.kinds(e).map((s: any) => s.name),
    names.map((name) => `Expressions.POSIX.ERE.${name}`))));
  assert(store.transaction(ERE.define)[0].get("POSIX_2024") === FS.find(ere_, "POSIX.1-2024")); // run again, it writes nothing
}

<a id="ERE-08"></a>
## ERE-08 · Printing: every kind as an ERE writes it, its special characters escaped, a lazy quantifier, a repeated one unambiguous; a BRE's term as a BRE

In [ ]:
{
  const every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0n), repeat(char("c"), 1n),
    repeat(char("d"), 0n, 1n), repeat(char("e"), 2n, 2n), repeat(char("f"), 2n), repeat(char("g"), 2n, 5n),
    cls([span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"), named("equivalence", "e"), char("]")], true),
    dot, anchor("end")));
  assert(text(every) === "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].$");
  assert(text(pattern(alt(repeat(char("x"), 0n, undefined, false), repeat(char("y"), 1n, 3n, false)))) === "x*?y{1,3}?");
  assert(text(pattern(alt(repeat(repeat(char("a"), 1n), 0n, 1n), repeat(anchor("start"), 0n)))) === "a+{0,1}^*"); // never `a+?`
  assert(text(pattern(alt(...[..."^.[$()|*+?{\\"].map((c) => char(c)), cls([char("("), char("\\")])))) === "\\^\\.\\[\\$\\(\\)\\|\\*\\+\\?\\{\\\\[(\\]");
  assert(text(repeat(char("a"), 0n, undefined, undefined, "{0,}")) === "a{0,}" && text(char("A", "\\A")) === "\\A"); // as written
  assert(RText.ToText(term(pattern(alt(repeat(char("a"), 1n), (u: any) => u.backreference((r: any) => r.ref(1n)))), B.DIALECT)) === "a\\{1,\\}\\1");
}

<a id="ERE-09"></a>
## ERE-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
{
  const interpreter = new FE.Interpreter(R.DIALECT, new Map(), { evaluated: new Set() }); // no kind evaluated, each refused by its dialect's name
  for (const spec of [patterns(abc), abc, char("a"), cls([]), anchor("end")]) {
    raises(NotImplementedError, () => interpreter.run(term(spec)), `POSIX ERE ${F.kindOf(term(spec))[0]} is not evaluated`);
  }
  raises(ValueError, () => interpreter.run(term(error("("))), "POSIX ERE error is not evaluated");
}

<a id="ERE-10"></a>
## ERE-10 · Completeness: grammars/posix-regex.json maps every kind, awk's escapes and POSIX.1-2024's repetition modifier among its entries

In [ ]:
{
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/posix-regex.json", "utf8"));
  const nodes: any[] = grammar.nodes;
  const mapped = new Set<string>(nodes.flatMap((n) => n.kinds ?? []));
  assert(equal([...mapped].filter((kind) => kind.startsWith("ERE.")).sort(), R.TAGS.map((tag) => `ERE.${tag}`).sort())); // every kind, exactly
  assert(equal([...mapped].sort(), [...B.TAGS.map((tag) => `BRE.${tag}`), ...R.TAGS.map((tag) => `ERE.${tag}`)].sort())); // and no other dialect's
  const by = new Map(nodes.map((n) => [n.node, n]));
  assert(same(by.get("escape sequence").kinds, ["ERE.character"]) && by.get("escape sequence").source.endsWith("XCU awk, Regular Expressions"));
  assert("not" in by.get("repetition modifier") && by.get("repetition modifier").source === "IEEE Std 1003.1-2024, XBD 9.4.6");
  assert(same(by.get("ERE_expression").kinds, ["ERE.assertion", "ERE.capturinggroup", "ERE.quantifier"]));
}